In [1]:
import pandas as pd
import numpy as np
from FeatureTransformer import FeatureTransformer
from anomaly_methods import AnomalyDetector

# Plotting Methods
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.graph_objects as go

# Method to plot data

In [2]:
def plot_attributes_plotly(df, attributes, x_col=None, start_index=100, title="Attribute Plot"):
    """
    Plots selected attributes from a DataFrame using Plotly.

    Parameters
    ----------
    df : pandas.DataFrame
        The input dataframe.
    attributes : list of str
        List of column names to plot.
    x_col : str, optional
        Column to use for the x-axis. If None, the index is used.
    title : str, optional
        Title of the plot.
    """
    fig = go.Figure()

    df_temp = df.iloc[start_index:]

    # Use index if no x_col provided
    x_vals = df_temp.index if x_col is None else df[x_col]

    for attr in attributes:
        if attr in df_temp.columns:
            fig.add_trace(
                go.Scatter(
                    x=x_vals,
                    y=df_temp[attr],
                    mode="lines",
                    name=attr,
                    hovertemplate="x: %{x}<br>y: %{y}<extra></extra>"
                )
            )
        else:
            print(f"Warning: '{attr}' not found in DataFrame columns.")

    fig.update_layout(
        title=title,
        xaxis_title=x_col if x_col else "Index",
        yaxis_title="Value",
        hovermode="x unified"
    )

    fig.show()

# Method to Print Anomaly Report

In [3]:
def analyze_anomaly_regime(
    mcd_res: tuple[np.ndarray, np.ndarray, np.ndarray],
    iso_res: tuple[np.ndarray, np.ndarray, np.ndarray],
    near_boundary_pct_threshold: float = 80.0
) -> dict:
    """
    Analyzes prediction outputs from FastMCD and Isolation Forest.
    
    Parameters:
        mcd_res: (is_anomaly, boundary_dist, percentile) from predict_fastmcd
        iso_res: (is_anomaly, boundary_dist, percentile) from predict_isolation_forest
        near_boundary_pct_threshold: Percentile rank threshold to flag 'getting close'
        
    Returns:
        dict containing consensus status, detailed metrics, and text analysis.
    """
    mcd_anom, mcd_dist, mcd_pct = int(mcd_res[0][0]), float(mcd_res[1][0]), float(mcd_res[2][0])
    iso_anom, iso_dist, iso_pct = int(iso_res[0][0]), float(iso_res[1][0]), float(iso_res[2][0])

    def _assess_model(is_anom: int, dist: float, pct: float, model_type: str) -> tuple[str, str]:
        if is_anom == 1:
            status = "ANOMALY"
            desc = f"Outside boundary ({dist:+.4f} units past cutoff, {pct:.1f}th pct)."
        elif pct >= near_boundary_pct_threshold or (model_type == "ISO" and pct <= (100 - near_boundary_pct_threshold)):
            status = "WARNING_NEAR_BOUNDARY"
            desc = f"Inside boundary but approaching tail ({dist:+.4f} units from cutoff, {pct:.1f}th pct)."
        else:
            status = "COMFORTABLY_INLIER"
            desc = f"Deep inside normal distribution ({dist:+.4f} units from cutoff, {pct:.1f}th pct)."
        return status, desc

    mcd_status, mcd_desc = _assess_model(mcd_anom, mcd_dist, mcd_pct, "MCD")
    iso_status, iso_desc = _assess_model(iso_anom, iso_dist, iso_pct, "ISO")

    # Consensus determination
    if mcd_anom == 1 and iso_anom == 1:
        consensus = "CONFIRMED_ANOMALY"
        summary = "High-confidence anomalous regime shift detected by both FastMCD and Isolation Forest."
    elif mcd_anom == 1 or iso_anom == 1:
        consensus = "DIVERGENT_ANOMALY"
        trigger = "FastMCD (Mahalanobis/Chi-Squared)" if mcd_anom == 1 else "Isolation Forest"
        summary = f"Divergence detected: Flagged as anomaly by {trigger} only."
    elif mcd_status == "WARNING_NEAR_BOUNDARY" or iso_status == "WARNING_NEAR_BOUNDARY":
        consensus = "APPROACHING_BOUNDARY"
        summary = "Not an anomaly, but feature vector is approaching the boundary threshold."
    else:
        consensus = "NORMAL_DISTRIBUTION"
        summary = "Comfortably inside the typical multivariate distribution."

    analysis_report = (
        f"--- ANOMALY REGIME ANALYSIS ---\n"
        f"Consensus: {consensus}\n"
        f"Summary:   {summary}\n\n"
        f"FastMCD (Chi-Squared Ellipsoid):\n"
        f"  • Status: {mcd_status}\n"
        f"  • Distance to Boundary: {mcd_dist:+.4f}\n"
        f"  • Percentile vs Train:   {mcd_pct:.2f}%\n"
        f"  • Detail: {mcd_desc}\n\n"
        f"Isolation Forest (Non-parametric Isolation):\n"
        f"  • Status: {iso_status}\n"
        f"  • Decision Score:       {iso_dist:+.4f}\n"
        f"  • Percentile vs Train:   {iso_pct:.2f}%\n"
        f"  • Detail: {iso_desc}\n"
        f"-------------------------------"
    )

    return {
        "consensus": consensus,
        "mcd": {"status": mcd_status, "distance": mcd_dist, "percentile": mcd_pct},
        "isolation_forest": {"status": iso_status, "distance": iso_dist, "percentile": iso_pct},
        "report": analysis_report
    }

# Read in data
## This dataframe just has OHLC for QQQ ETF

In [4]:
df = pd.read_csv("../data/QQQ.csv")

# Compute Moving Averages
## We will compute moving averages in windows of 10, 20, 50

## First we will create a new object of the FeatureTransformer Class

In [5]:
ft = FeatureTransformer(df) # New Object of Feature Transformer

# Compute Moving Averages, traditionally we use "Close" in trading to compute a moving average
# We will use Close here as the feature
# Compute Moving Averages (EMA & SMA) (You can adjust this to be whatever Moving Averages you wish)
for w in [10, 20, 50]:
    df = ft.add_moving_averages(['Close'], w)

## Visualize Close along with the Moving Averages to see if we can spot stretches

In [6]:
plot_attributes_plotly(df, ['Close', 'Close_ema_10', 'Close_sma_10',
       'Close_ema_20', 'Close_sma_20', 'Close_ema_50', 'Close_sma_50'])

# Stationarity & Feature Engineering

Since raw security prices are non-stationary, anomaly detection models require scale-invariant features to generalize across different assets and price regimes.

## 1. Price-to-Moving Average Deviations
To identify stretched prices, we compute the percentage distance between the closing price and its respective Simple Moving Average (SMA):
$$\text{Price Deviation} = \frac{\text{Close} - \text{SMA}}{\text{SMA}}$$

## 2. Exponential vs. Simple Moving Average Spread
As price momentum accelerates, short-term EMAs diverge rapidly from longer-term SMAs:
$$\text{EMA Spread} = {\text{EMA} - \text{SMA}}$$

We leverage machine learning models on these stationary percentage spreads to detect structural price stretches that signal potential trend exhaustion and mean-reversion setups.

In [7]:
plot_attributes_plotly(df, ['Close_ema_10', 'Close_sma_10',
       'Close_sma_20', 'Close_ema_20', 'Close_sma_50', 'Close_ema_50'])

In [8]:
# Create Stationary Features for ['Close_ema_10','Close_sma_20', 'Close_ema_20', 'Close_sma_50', 'Close_ema_50']
features_toScale = ['Close_ema_10', 'Close_sma_10', 'Close_sma_20', 'Close_ema_20', 'Close_sma_50', 'Close_ema_50']

for feat in features_toScale:
    df[f"{feat}_scaled"] = (df['Close'] - df[feat])/df[feat]

# Compute the differences of EMA to SMA 
# #We will use that in addition to the percent differences of Close to the respective MA
df['ema_sma_10_difference'] = df['Close_ema_10_scaled'] - df['Close_sma_10_scaled']
df['ema_sma_20_difference'] = df['Close_ema_20_scaled'] - df['Close_sma_20_scaled']
df['ema_sma_50_difference'] = df['Close_ema_50_scaled'] - df['Close_sma_50_scaled']

## Plot the newly scaled features

### Now we can see that there is some context to these features, they have become more bounded regardless of what the price of the underlying security is

In [9]:
plot_attributes_plotly(df, ['Close_ema_10_scaled', 'Close_sma_20_scaled', 'Close_ema_20_scaled',
       'Close_sma_50_scaled', 'Close_ema_50_scaled'])

In [10]:
plot_attributes_plotly(df, [ 'ema_sma_50_difference', 'ema_sma_20_difference', 'ema_sma_10_difference'])

# Feature Correlation & Collinearity Analysis

## Collinearity in Moving Averages
High pairwise correlation is expected when analyzing raw moving average values (such as SMAs and EMAs over similar windows), as they reflect the same underlying price series.

## Signal Extraction via Spreads
Our target signals are derived from the relative **differences and percentage spreads** between these indicators (e.g., $\text{EMA} - \text{SMA}$ or $\frac{\text{EMA} - \text{SMA}}{\text{SMA}}$). Discarding collinear price-tracking features is not required here, as taking their differential transforms raw price levels into stationary momentum features.

In [11]:
df[[ 'Close_ema_10_scaled', 'Close_sma_20_scaled', 'Close_ema_20_scaled',
       'Close_sma_50_scaled', 'Close_ema_50_scaled', 'Close_sma_10_scaled',
       'ema_sma_10_difference', 'ema_sma_20_difference',
       'ema_sma_50_difference']].corr()

,Close_ema_10_scaled,Close_sma_20_scaled,Close_ema_20_scaled,Close_sma_50_scaled,Close_ema_50_scaled,Close_sma_10_scaled,ema_sma_10_difference,ema_sma_20_difference,ema_sma_50_difference
Close_ema_10_scaled,1.000000,0.928467,0.941771,0.698140,0.739447,0.980455,-0.559890,-0.507124,-0.233732
Close_sma_20_scaled,0.928467,1.000000,0.981951,0.816057,0.840431,0.877861,-0.382759,-0.672553,-0.368360
Close_ema_20_scaled,0.941771,0.981951,1.000000,0.880483,0.900101,0.889504,-0.384692,-0.520446,-0.423504
Close_sma_50_scaled,0.698140,0.816057,0.880483,1.000000,0.981300,0.612157,-0.084899,-0.239833,-0.643784
Close_ema_50_scaled,0.739447,0.840431,0.900101,0.981300,1.000000,0.660653,-0.143023,-0.272483,-0.484457
Close_sma_10_scaled,0.980455,0.877861,0.889504,0.612157,0.660653,1.000000,-0.711962,-0.483662,-0.157183
ema_sma_10_difference,-0.559890,-0.382759,-0.384692,-0.084899,-0.143023,-0.711962,1.000000,0.226655,-0.173076
ema_sma_20_difference,-0.507124,-0.672553,-0.520446,-0.239833,-0.272483,-0.483662,0.226655,1.000000,0.006294
ema_sma_50_difference,-0.233732,-0.368360,-0.423504,-0.643784,-0.484457,-0.157183,-0.173076,0.006294,1.000000


In [12]:
df[[ 'Close_ema_10_scaled', 'Close_sma_20_scaled', 'Close_ema_20_scaled',
       'Close_sma_50_scaled', 'Close_ema_50_scaled', 'Close_sma_10_scaled',
       'ema_sma_10_difference', 'ema_sma_20_difference',
       'ema_sma_50_difference']].describe()

,Close_ema_10_scaled,Close_sma_20_scaled,Close_ema_20_scaled,Close_sma_50_scaled,Close_ema_50_scaled,Close_sma_10_scaled,ema_sma_10_difference,ema_sma_20_difference,ema_sma_50_difference
count,6813.000000,6794.000000,6813.000000,6764.000000,6813.000000,6804.000000,6804.000000,6794.000000,6764.000000
mean,0.001683,0.003934,0.003602,0.010270,0.009399,0.001872,-0.000190,-0.000360,-0.000952
std,0.021867,0.037080,0.032124,0.060176,0.052516,0.025813,0.006129,0.008213,0.013240
min,-0.160523,-0.234022,-0.203211,-0.326340,-0.298246,-0.173415,-0.041841,-0.058458,-0.080619
25%,-0.008353,-0.013576,-0.010731,-0.016493,-0.012715,-0.010404,-0.003163,-0.004247,-0.006675
50%,0.004692,0.008853,0.009062,0.019341,0.018896,0.004737,0.000188,0.000302,0.000116
75%,0.014018,0.025854,0.022838,0.045900,0.041217,0.016287,0.003057,0.004089,0.006150
max,0.143672,0.201747,0.157930,0.246991,0.211354,0.180179,0.032227,0.036479,0.048664


# Prep our features for Anomaly Detection

## We need to scale our features so each are on the same playing field

In [13]:
# Define our input features
input_features = ['Close_ema_10_scaled', 'Close_sma_20_scaled', 'Close_ema_20_scaled',
       'Close_sma_50_scaled', 'Close_ema_50_scaled', 'Close_sma_10_scaled',
       'ema_sma_10_difference', 'ema_sma_20_difference',
       'ema_sma_50_difference']

In [16]:
# Index to try out
start_index = 50 # Since 50 MA is our largest moving average, the 50th index is our first training period
_index = 5289
end_idx = _index - 30 # We don't want to include our test point, we will leave a 30 period buffer from test point to where training ends

X_train_scaled, scaler = ft.scale_features(input_features, start_index, end_idx) # Scaling our features using standard scaler

# Fit FastMCD Model
ad = AnomalyDetector()
fast_mcd_model, mcd_boundary_dist = ad.fit_fastmcd(X_train_scaled)


# predict
## Scale test point
X_new = df.iloc[_index][input_features].to_numpy().reshape(1,-1)

X_test_scaled = scaler.transform(X_new)

## Fast MCD
is_anomaly_mcd, boundary_dist_mcd, percentiles_mcd = ad.predict_fastmcd(X_test_scaled)

## Isolation Forest
iso_model, iso_train_boundary_dists = ad.fit_isolation_forest(X_train_scaled)

## Predict if an anomaly exists on Isolation Forest
iso_is_anomaly, iso_boundary_dist, iso_percentile = ad.predict_isolation_forest(X_test_scaled)

/usr/local/python/3.14.2/lib/python3.14/site-packages/sklearn/utils/validation.py:2830: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(


In [17]:
# Pass your outputs directly into the analyzer
analysis = analyze_anomaly_regime(
    mcd_res=(is_anomaly_mcd, boundary_dist_mcd, percentiles_mcd),
    iso_res=(iso_is_anomaly, iso_boundary_dist, iso_percentile),
    near_boundary_pct_threshold=80.0
)

print(analysis["report"])

--- ANOMALY REGIME ANALYSIS ---
Consensus: CONFIRMED_ANOMALY
Summary:   High-confidence anomalous regime shift detected by both FastMCD and Isolation Forest.

FastMCD (Chi-Squared Ellipsoid):
  • Status: ANOMALY
  • Distance to Boundary: +20.7453
  • Percentile vs Train:   93.59%
  • Detail: Outside boundary (+20.7453 units past cutoff, 93.6th pct).

Isolation Forest (Non-parametric Isolation):
  • Status: ANOMALY
  • Decision Score:       -0.1142
  • Percentile vs Train:   2.53%
  • Detail: Outside boundary (-0.1142 units past cutoff, 2.5th pct).
-------------------------------
